In [1]:
"""
NOAA CDO API - Hourly weather for a zip code on a given date.
Free token: https://www.ncdc.noaa.gov/cdo-web/token
"""

import requests
import pandas as pd

NOAA_TOKEN = "BzDPejkQUcLmyFsHAHaArxcfQSfTCBYp"
BASE_URL   = "https://www.ncdc.noaa.gov/cdo-web/api/v2"
HEADERS    = {"token": NOAA_TOKEN}

ZIP_CODE   = "39501"        # Gulfport, MS
DATE       = "2024-12-07"

# ── Step 1: Find nearest station for the zip code ─────────────────────────────

resp = requests.get(f"{BASE_URL}/stations", headers=HEADERS, params={
    "datasetid":  "LCD",
    "locationid": f"ZIP:{ZIP_CODE}",
    "startdate":  DATE,
    "enddate":    DATE,
    "limit":      5,
})
resp.raise_for_status()

stations = resp.json().get("results", [])
station_id   = stations[0]["id"]
station_name = stations[0]["name"]
print(f"Station: {station_name} ({station_id})")

# ── Step 2: Pull hourly data for that station ─────────────────────────────────

resp = requests.get(f"{BASE_URL}/data", headers=HEADERS, params={
    "datasetid": "LCD",
    "stationid": station_id,
    "startdate": DATE,
    "enddate":   DATE,
    "units":     "standard",
    "limit":     1000,
})
resp.raise_for_status()

raw = resp.json().get("results", [])
print(f"Records returned: {len(raw)}")

# ── Step 3: Pivot to one row per hour ─────────────────────────────────────────

df = pd.DataFrame(raw)

df = df.pivot_table(
    index=["station", "date"],
    columns="datatype",
    values="value",
    aggfunc="first"
).reset_index()
df.columns.name = None

df = df.rename(columns={
    "date":                          "datetime",
    "HourlyDryBulbTemperature":      "temp_f",
    "HourlyRelativeHumidity":        "humidity_pct",
    "HourlyDewPointTemperature":     "dewpoint_f",
    "HourlyWindSpeed":               "wind_speed_mph",
    "HourlyWindDirection":           "wind_dir",
    "HourlyPrecipitation":           "precip_in",
    "HourlySkyConditions":           "sky",
})

df["datetime"] = pd.to_datetime(df["datetime"])

for col in ["temp_f", "humidity_pct", "dewpoint_f", "wind_speed_mph", "precip_in"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ── Step 4: Preview ───────────────────────────────────────────────────────────

cols = [c for c in ["datetime", "temp_f", "humidity_pct", "wind_speed_mph", "precip_in"] if c in df.columns]
print(df[cols].to_string(index=False))

Station: GULFPORT BILOXI AIRPORT, MS US (WBAN:93874)


HTTPError: 500 Server Error:  for url: https://www.ncdc.noaa.gov/cdo-web/api/v2/data?datasetid=LCD&stationid=WBAN%3A93874&startdate=2024-12-07&enddate=2024-12-07&units=standard&limit=1000